# UCS420 – Lab Assignment 10
## NLP using Python-II

## Q1. Text preprocessing and word frequency

In [ ]:
import re
import nltk
from collections import Counter
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords

for resource in ["punkt", "punkt_tab", "stopwords"]:
    nltk.download(resource, quiet=True)

paragraph = """Artificial intelligence is transforming the way people learn, work, and solve problems. I enjoy exploring how intelligent systems can understand language, recognize patterns, and support creative ideas. Machine learning allows computers to improve through experience and data. Natural language processing makes communication between humans and computers more useful. I find this field exciting because it combines mathematics, programming, and real-world problem solving."""

clean_text = re.sub(r"[^a-zA-Z0-9\s]", "", paragraph.lower())
words_split = clean_text.split()
words_nltk = word_tokenize(clean_text)
sentences = sent_tokenize(paragraph)

stop_words = set(stopwords.words("english"))
filtered_words = [word for word in words_nltk if word.isalpha() and word not in stop_words]
frequency = Counter(filtered_words)

print("Original paragraph:\n", paragraph)
print("\nLowercase text without punctuation:\n", clean_text)
print("\nSentences:")
for sentence in sentences:
    print("-", sentence)
print("\nPython split():", words_split)
print("\nNLTK word_tokenize():", words_nltk)
print("\nStopword-removed tokens:", filtered_words)
print("\nWord frequency distribution:")
print(frequency.most_common())

Python `split()` separates text at whitespace, while NLTK's `word_tokenize()` uses language-aware tokenization and can separate punctuation into individual tokens. Since punctuation was removed before tokenization here, the resulting word lists are very similar.

## Q2. Regular expressions, stopword removal, stemming and lemmatization

In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

alphabetic_words = re.findall(r"\b[a-zA-Z]+\b", paragraph.lower())
filtered_q2 = [word for word in alphabetic_words if word not in stop_words]

porter = PorterStemmer()
lemmatizer = WordNetLemmatizer()

stemmed_words = [porter.stem(word) for word in filtered_q2]
lemmatized_words = [lemmatizer.lemmatize(word) for word in filtered_q2]

print("Words extracted using re.findall():", alphabetic_words)
print("\nAfter stopword removal:", filtered_q2)
print("\nStemmed words:", stemmed_words)
print("\nLemmatized words:", lemmatized_words)

Stemming mechanically removes word endings and may produce roots that are not valid words. Lemmatization uses vocabulary and morphological analysis to return dictionary forms. Stemming is useful when speed and broad matching matter; lemmatization is preferable when readable, linguistically meaningful words are important.

## Q3. Bag of Words and TF-IDF

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import numpy as np

texts = [
    "Electric vehicles reduce air pollution and use rechargeable batteries.",
    "Solar energy produces clean electricity from sunlight.",
    "Artificial intelligence helps computers learn patterns and automate tasks."
]

count_vectorizer = CountVectorizer()
bow_matrix = count_vectorizer.fit_transform(texts)

print("Bag of Words feature names:")
print(count_vectorizer.get_feature_names_out())
print("\nBag of Words matrix:")
print(bow_matrix.toarray())

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(texts)
features = tfidf_vectorizer.get_feature_names_out()

print("\nTF-IDF matrix:")
print(np.round(tfidf_matrix.toarray(), 3))

for i, text in enumerate(texts):
    row = tfidf_matrix[i].toarray().ravel()
    top_indices = row.argsort()[-3:][::-1]
    print(f"\nText {i + 1}: {text}")
    print("Top 3 TF-IDF keywords:", [(features[j], round(row[j], 3)) for j in top_indices])

TF-IDF gives higher importance to terms that are frequent in one text but less common across the collection. The top keywords therefore help distinguish the subject of each short text.

## Q4. Similarity between two technologies

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

technology_texts = [
    """Artificial intelligence enables machines to perform tasks that usually require human intelligence.
It uses data and algorithms to recognize patterns and make predictions.
AI is applied in healthcare, education, transportation, and automation.
Machine learning is an important method used to build many AI systems.""",
    """Blockchain is a distributed digital ledger that stores records across a network.
It uses cryptographic techniques to make stored information difficult to alter.
Blockchain supports transparent transactions without relying on one central authority.
It is commonly used in digital currencies, supply chains, and smart contracts."""
]

def preprocess(text):
    return re.findall(r"\b[a-zA-Z]+\b", text.lower())

tokens_1 = preprocess(technology_texts[0])
tokens_2 = preprocess(technology_texts[1])

set_1 = set(tokens_1)
set_2 = set(tokens_2)
jaccard = len(set_1 & set_2) / len(set_1 | set_2)

vectorizer = TfidfVectorizer()
tfidf_values = vectorizer.fit_transform(technology_texts)
cosine = cosine_similarity(tfidf_values[0:1], tfidf_values[1:2])[0, 0]

print("Technology 1 tokens:", tokens_1)
print("\nTechnology 2 tokens:", tokens_2)
print("\nJaccard similarity:", round(jaccard, 4))
print("Cosine similarity:", round(float(cosine), 4))

Jaccard similarity compares the overlap between unique words and ignores how often words occur. Cosine similarity compares TF-IDF-weighted vectors, so it considers the importance of terms within each text. For these descriptions, cosine similarity generally provides more informative insight because it accounts for term importance, while both measures indicate that the technologies discuss substantially different concepts.

## Q5. Sentiment analysis and word cloud

In [ ]:
from textblob import TextBlob
from wordcloud import WordCloud
import matplotlib.pyplot as plt

reviews = [
    "The product is excellent, easy to use, and works perfectly.",
    "I am very happy with the quality and the fast delivery.",
    "The service was disappointing and the product stopped working.",
    "The item is poorly made and customer support was unhelpful.",
    "The product is okay and performs as expected.",
    "Delivery was average and the packaging was neither good nor bad."
]

sentiment_results = []
for review in reviews:
    analysis = TextBlob(review).sentiment
    polarity = analysis.polarity
    subjectivity = analysis.subjectivity
    if polarity > 0:
        label = "Positive"
    elif polarity < 0:
        label = "Negative"
    else:
        label = "Neutral"
    sentiment_results.append((review, polarity, subjectivity, label))

for review, polarity, subjectivity, label in sentiment_results:
    print("Review:", review)
    print("Polarity:", round(polarity, 3))
    print("Subjectivity:", round(subjectivity, 3))
    print("Classification:", label)
    print()

positive_reviews = [review for review, polarity, subjectivity, label in sentiment_results if label == "Positive"]
positive_text = " ".join(positive_reviews)

if positive_text.strip():
    wordcloud = WordCloud(width=900, height=450, background_color="white").generate(positive_text)
    plt.figure(figsize=(12, 6))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.show()
else:
    print("No positive reviews available for the word cloud.")

Polarity ranges from -1 (negative) to +1 (positive), while subjectivity ranges from 0 (objective) to 1 (subjective). The word cloud visualizes frequently occurring terms in the positive reviews.

## Q6. Text generation using an LSTM

In [ ]:
import numpy as np

try:
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Embedding, LSTM, Dense
except ImportError:
    from keras.preprocessing.text import Tokenizer
    from keras.preprocessing.sequence import pad_sequences
    from keras.models import Sequential
    from keras.layers import Embedding, LSTM, Dense

training_text = """Technology continues to change the world around us. Artificial intelligence helps people analyze information and solve difficult problems. Learning programming develops logical thinking and creativity. Data provides useful patterns that support better decisions. Computers can process large amounts of information quickly. Responsible innovation should improve education healthcare communication and everyday life. Students learn new skills through practice curiosity and consistent effort. Modern software connects people and makes collaboration easier. Scientific research creates knowledge that can guide future inventions. A thoughtful approach to technology can create a more useful and inclusive world."""

tokenizer = Tokenizer()
tokenizer.fit_on_texts([training_text])
total_words = len(tokenizer.word_index) + 1

input_sequences = []
for line in training_text.split("."):
    token_list = tokenizer.texts_to_sequences([line.strip()])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i + 1])

max_sequence_len = max(len(sequence) for sequence in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_sequence_len, padding="pre")
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

model = Sequential([
    Embedding(total_words, 32, input_length=max_sequence_len - 1),
    LSTM(64),
    Dense(total_words, activation="softmax")
])
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.fit(X, y, epochs=150, verbose=0)

def generate_text(seed_text, next_words=12):
    result = seed_text
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([result])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len - 1, padding="pre")
        predicted = int(np.argmax(model.predict(token_list, verbose=0), axis=-1)[0])
        output_word = tokenizer.index_word.get(predicted, "")
        if not output_word:
            break
        result += " " + output_word
    return result

for seed in ["technology", "students", "learning"]:
    print(generate_text(seed, 12).capitalize() + ".")

The LSTM is trained to predict the next word from preceding words in the training paragraph. The generated lines are illustrative; with a small training corpus, the model may repeat phrases or produce grammatically imperfect text.